In [1]:
from keras.applications import VGG16

# Load the VGG16 model without the top (fully connected layers)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

58889256/58889256 [==============================] - 0s 0us/step


In [2]:
for layer in base_model.layers:
    layer.trainable = False

In [3]:
from keras.models import Model
from keras.layers import Flatten, Dense

# Add custom regression layers on top of VGG16
x = base_model.output
x = Flatten()(x)
x = Dense(1024, activation='relu')(x)
output = Dense(1, activation='linear')(x)  # Output layer for regression, adjust units for your task

# Create a new model
model = Model(inputs=base_model.input, outputs=output)


In [4]:
model.compile(optimizer='adam', loss='mean_squared_error')


# Data creation

In [1]:
import pandas as pd
import os
import csv
import requests
import time

In [2]:
# Load df
df = pd.read_csv("/notebooks/sample_products.csv")

In [3]:
# Get ids and urls and zip them
ids = df.id.values.tolist()
urls = df.image_url.values.tolist()
all_values = list(zip(ids, urls))

In [4]:
# Function to download images



def image_extractor(ids_and_images_url):
    # Check if images folder exists, if not create it
    if not os.path.exists('images'):
        os.makedirs('images')

    # Create a CSV file to store image names and IDs
    csv_file_path = 'images/images.csv'
    with open(csv_file_path, 'w', newline='') as csvfile:
        fieldnames = ['id', 'image_name']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        total_images_downloaded = 0

        # Download the images using the requests library
        for id, image_url in ids_and_images_url:
            # Strip leading and trailing whitespace from the URL
            image_url = image_url.strip()

            try:
                headers = {
                          "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36"
                          }
                response = requests.get(image_url, timeout=1, headers=headers)
                response.raise_for_status()

                # Check if the request was successful
                if response.status_code == 200:
                    # Save the image to a file
                    img_file = f'{id}.jpg'
                    with open(f'images/{img_file}', 'wb') as f:
                        f.write(response.content)

                    # Write image name and ID to CSV
                    writer.writerow({'id': id, 'image_name': img_file})
                    total_images_downloaded += 1

                    # Print progress
                    print(f"Downloaded {total_images_downloaded} images")

            except (requests.exceptions.RequestException, ConnectionError) as e:
                print(f"Connection error encountered for ID {id}: {e}")

            # Add a small delay between requests to avoid overwhelming the server
            time.sleep(0.5)

    print("Download process completed.")






## image downloading process has some errors. Get the output of the image extractor and locate the mages with errors for data pre-processing.

In [9]:
# Extract images into images folder
image_extractor(all_values)

Downloaded 1 images
Downloaded 2 images
Downloaded 3 images
Downloaded 4 images
Downloaded 5 images
Downloaded 6 images
Downloaded 7 images
Downloaded 8 images
Downloaded 9 images
Downloaded 10 images
Downloaded 11 images
Downloaded 12 images
Downloaded 13 images
Downloaded 14 images
Downloaded 15 images
Downloaded 16 images
Downloaded 17 images
Downloaded 18 images
Downloaded 19 images
Downloaded 20 images
Downloaded 21 images
Downloaded 22 images
Downloaded 23 images
Downloaded 24 images
Downloaded 25 images
Downloaded 26 images
Downloaded 27 images
Downloaded 28 images
Downloaded 29 images
Downloaded 30 images
Downloaded 31 images
Downloaded 32 images
Downloaded 33 images
Downloaded 34 images
Downloaded 35 images
Downloaded 36 images
Downloaded 37 images
Downloaded 38 images
Downloaded 39 images
Downloaded 40 images
Downloaded 41 images
Downloaded 42 images
Downloaded 43 images
Downloaded 44 images
Downloaded 45 images
Downloaded 46 images
Downloaded 47 images
Downloaded 48 images
D

In [5]:
# Combine sample products and csv
images_df = pd.read_csv('/notebooks/images/images.csv')

# Merge DataFrames on the 'id' column
merged_df = pd.merge(df, images_df, on='id', how='inner')

In [6]:
# view data
pd.set_option('display.max_colwidth', None)

merged_df

,Unnamed: 0,id,product title,image_url,price,image_name
0,0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black, Checked-Large 29-Inch",https://m.media-amazon.com/images/I/815dLQKYIYL._AC_UL320_.jpg,139.99,B014TMV5YE.jpg
1,1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcase Double Wheels TSA Lock Blue,https://m.media-amazon.com/images/I/81bQlm7vf6L._AC_UL320_.jpg,169.99,B07GDLCQXV.jpg
2,2,B07XSCCZYG,"Platinum Elite Softside Expandable Checked Luggage, 8 Wheel Spinner Suitcase, TSA Lock, Men and Women, True Navy Blue, Checked Medium 25-Inch",https://m.media-amazon.com/images/I/71EA35zvJBL._AC_UL320_.jpg,365.49,B07XSCCZYG.jpg
3,3,B08MVFKGJM,"Freeform Hardside Expandable with Double Spinner Wheels, Navy, 2-Piece Set (21/28)",https://m.media-amazon.com/images/I/91k6NYLQyIL._AC_UL320_.jpg,291.59,B08MVFKGJM.jpg
4,4,B01DJLKZBA,"Winfield 2 Hardside Expandable Luggage with Spinner Wheels, Checked-Large 28-Inch, Deep Blue",https://m.media-amazon.com/images/I/61NJoaZcP9L._AC_UL320_.jpg,174.99,B01DJLKZBA.jpg
...,...,...,...,...,...,...
9991,9995,B0BR617B8P,Mens Athletic Workout Shorts with Compression Liner 7 inch Inseam,https://m.media-amazon.com/images/I/61MFjWYxa5L._AC_UL320_.jpg,34.99,B0BR617B8P.jpg
9992,9996,B07VWSP5HD,Men's Knitted Regular Fit Full Zip Cardigan Sweater with Soft Brushed Flannel Lining,https://m.media-amazon.com/images/I/91Hm4RVlu8L._AC_UL320_.jpg,49.99,B07VWSP5HD.jpg
9993,9997,B0CD6MBV8T,Men's Jacket Windproof Qulited Bomber Jackets Casual Fall Winter Warm Padded Coats Outwear,https://m.media-amazon.com/images/I/61sVoWGbg3L._AC_UL320_.jpg,46.98,B0CD6MBV8T.jpg
9994,9998,B08XW98F22,Men's Dry Franchise Polo,https://m.media-amazon.com/images/I/51yJ8ZYPcsL._AC_UL320_.jpg,46.87,B08XW98F22.jpg


In [7]:
merged_df.to_csv('merged_data_myself.csv', index=False)

In [8]:
merged_df.drop(columns=['Unnamed: 0'], inplace=True)


In [9]:
merged_df.drop(columns=['id', 'product title', 'image_url'], inplace=True)

In [10]:
merged_df

,price,image_name
0,139.99,B014TMV5YE.jpg
1,169.99,B07GDLCQXV.jpg
2,365.49,B07XSCCZYG.jpg
3,291.59,B08MVFKGJM.jpg
4,174.99,B01DJLKZBA.jpg
...,...,...
9991,34.99,B0BR617B8P.jpg
9992,49.99,B07VWSP5HD.jpg
9993,46.98,B0CD6MBV8T.jpg
9994,46.87,B08XW98F22.jpg


In [11]:
for i, row in merged_df.iterrows():
    if row['price'] == 0.00:
        merged_df.drop(i, inplace=True)

In [13]:
merged_df

,price,image_name
0,139.99,B014TMV5YE.jpg
1,169.99,B07GDLCQXV.jpg
2,365.49,B07XSCCZYG.jpg
3,291.59,B08MVFKGJM.jpg
4,174.99,B01DJLKZBA.jpg
...,...,...
9990,209.50,B09V11RVFC.jpg
9991,34.99,B0BR617B8P.jpg
9992,49.99,B07VWSP5HD.jpg
9993,46.98,B0CD6MBV8T.jpg


In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import SGD

In [31]:
# Load your dataset (assuming it's in a DataFrame format)
df = merged_df

In [32]:
# Split dataset into train and test sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [33]:
# Set up ImageDataGenerator for loading images
datagen = ImageDataGenerator(rescale=1./255)  # Scale pixel values to [0, 1]

In [34]:
# Define image size
img_size = (224, 224)  # VGG16 input size

In [35]:
# Load and preprocess images using the ImageDataGenerator
train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/notebooks/images',
    x_col='image_name',  # Assuming 'image_name' column contains file names
    y_col='price',  # Assuming 'price' column contains continuous labels
    target_size=img_size,
    batch_size=32,
    class_mode='raw'  # Use 'raw' for regression
)

test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/notebooks/images',
    x_col='image_name',
    y_col='price',
    target_size=img_size,
    batch_size=32,
    class_mode='raw'
)

Found 7851 validated image filenames.
Found 1963 validated image filenames.


In [36]:
# Load pre-trained VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the convolutional base
base_model.trainable = False

# Create a new model on top
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(1)  # Output layer with a single neuron for regression
])

In [37]:
# Compile the model
model.compile(optimizer=SGD(lr=0.001), loss='mean_squared_error', metrics=['mae'])

/usr/local/lib/python3.9/dist-packages/keras/optimizers/optimizer_v2/gradient_descent.py:108: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [38]:
# Train the model
model.fit(train_generator, epochs=10, validation_data=test_generator)

Epoch 1/10
246/246 [==============================] - 64s 234ms/step - loss: 26365.3867 - mae: 60.2322 - val_loss: 7849.3101 - val_mae: 40.4084
Epoch 2/10
246/246 [==============================] - 47s 191ms/step - loss: 5601.6841 - mae: 32.2409 - val_loss: 6914.7139 - val_mae: 33.0640
Epoch 3/10
246/246 [==============================] - 37s 150ms/step - loss: 5062.9951 - mae: 29.8682 - val_loss: 6539.5859 - val_mae: 33.5078
Epoch 4/10
246/246 [==============================] - 32s 131ms/step - loss: 4863.3945 - mae: 31.2758 - val_loss: 6389.0010 - val_mae: 35.1351
Epoch 5/10
246/246 [==============================] - 33s 132ms/step - loss: 4788.8813 - mae: 32.8303 - val_loss: 6323.3623 - val_mae: 36.4195
Epoch 6/10
246/246 [==============================] - 33s 132ms/step - loss: 4761.2969 - mae: 33.9731 - val_loss: 6294.8149 - val_mae: 37.2443
Epoch 7/10
246/246 [==============================] - 32s 131ms/step - loss: 4751.0747 - mae: 34.6490 - val_loss: 6280.8447 - val_mae: 37.770

# ResNet50

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import SGD

# Load your dataset (assuming it's in a DataFrame format)
df = merged_df

# Split dataset into train and test sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Set up ImageDataGenerator for loading images
datagen = ImageDataGenerator(rescale=1./255)  # Scale pixel values to [0, 1]

# Define image size
img_size = (224, 224)  # ResNet50 input size

# Load and preprocess images using the ImageDataGenerator
train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/notebooks/images',
    x_col='image_name',  # Assuming 'image_name' column contains file names
    y_col='price',  # Assuming 'price' column contains continuous labels
    target_size=img_size,
    batch_size=32,
    class_mode='raw'  # Use 'raw' for regression
)

test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/notebooks/images',
    x_col='image_name',
    y_col='price',
    target_size=img_size,
    batch_size=32,
    class_mode='raw'
)

# Load pre-trained ResNet50 model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the convolutional base
base_model.trainable = False

# Create a new model on top
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(1)  # Output layer with a single neuron for regression
])

# Compile the model
model.compile(optimizer=SGD(lr=0.001), loss='mean_squared_error', metrics=['mae'])

# Train the model
model.fit(train_generator, epochs=20, validation_data=test_generator)


Found 7851 validated image filenames.
Found 1963 validated image filenames.


/usr/local/lib/python3.9/dist-packages/keras/optimizers/optimizer_v2/gradient_descent.py:108: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


Epoch 1/20
110/246 [============>.................] - ETA: 47s - loss: nan - mae: nan

KeyboardInterrupt: 

# without image rescale

In [3]:
pip install ultralytics==8.0.196

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.1/631.1 kB 38.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="cbvlBgaQv3TtG6UGBXbL")
project = rf.workspace("objectdetecttrade").project("fxtb")
dataset = project.version(661).download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


transfer learning (BCNet)
models (3E-Net)
EfficientNetB0
EfficientNetV2B0
EfficientNetV2B0-21k
ResNetV1-50
ResNetV2-50
MobileNetV1
MobileNetV2

additionally, search CNN in paperswithcode

https://paperswithcode.com/paper/libra-r-cnn-towards-balanced-learning-for

https://paperswithcode.com/paper/cascade-r-cnn-delving-into-high-quality

https://paperswithcode.com/paper/dynamic-r-cnn-towards-high-quality-object

https://paperswithcode.com/paper/mask-r-cnn

https://paperswithcode.com/paper/grid-r-cnn